# Build Train-Only Sentiment Features on Colab GPU

Use this notebook for the expensive `Movies_and_TV` / `Video_Games` review-text sentiment pass. It can also run an optional fixed-alpha evaluation sweep after sentiment artifacts exist.

Default workflow:

```bash
python -m src.features.build_sentiment_features --dataset movies_and_tv
```

Optional evaluation workflow:

```bash
python -m src.evaluation.evaluate --dataset movies_and_tv --no-knn --advanced --alpha 0.6
```

Sentiment writes local/reproducible artifacts under `data/processed/<dataset>/advanced_features/`:

- `train_sentiment.parquet`
- `train_sentiment_meta.json`
- `user_review_aggregates.parquet`
- `item_review_aggregates.parquet`

The optional alpha sweep writes `data/processed/<dataset>/metrics_alpha_sweep.json`. Do not commit any generated data artifacts.


## 1. Select GPU Runtime

In Colab: `Runtime` -> `Change runtime type` -> `T4 GPU`.

The repo code prefers `cuda -> mps -> cpu`, so Colab should report `cuda` in `train_sentiment_meta.json`.

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 2. Mount Drive and Point at Data

Put your data in Drive with this shape:

```text
MyDrive/amazon-hybrid-recsys-data/
  raw/
    Movies_and_TV.jsonl or Movies_and_TV.jsonl.gz
    meta_Movies_and_TV.jsonl or meta_Movies_and_TV.jsonl.gz
  processed/
    movies_and_tv/
      train.parquet
      test.parquet
      metadata.parquet
```

The notebook symlinks the cloned repo's `data/raw` and `data/processed` to this Drive folder, so outputs persist after the Colab VM shuts down.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/amazon-hybrid-recsys-data'
DATASET = 'movies_and_tv'  # change to 'video_games' if needed
BRANCH = 'feat/advanced-models'

# Sentiment controls
RUN_SMOKE_SENTIMENT = True
RUN_FULL_SENTIMENT = True

# Optional evaluation controls. Keep capped by default for Movies_and_TV.
RUN_ALPHA_SWEEP = False
ALPHAS = [0.75, 0.6, 0.5, 0.4]
MAX_EVAL_USERS = 5000
MAX_TEST_ROWS = 100000


## 3. Clone Repo and Install Runtime Dependencies

This installs the project requirements so both sentiment feature building and optional evaluation can run.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def run(*args):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['PYTHONIOENCODING'] = 'utf-8'
    print('+', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(
        [str(arg) for arg in args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=0,
        env=env,
    )
    assert process.stdout is not None
    while True:
        chunk = process.stdout.read(1)
        if chunk == '' and process.poll() is not None:
            break
        if chunk:
            print(chunk, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, [str(arg) for arg in args])


REPO = Path('/content/amazon-hybrid-recsys')
if not REPO.exists():
    run('git', 'clone', 'https://github.com/Fgram-devAI/amazon-hybrid-recsys.git', REPO)
os.chdir(REPO)
print('cwd:', Path.cwd())
run('git', 'fetch', 'origin', BRANCH)
run('git', 'checkout', BRANCH)
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt')

## 4. Link Drive Data Into the Repo

This does not copy the large files. It creates symlinks from repo-local `data/raw` and `data/processed` to your Drive-backed folders.

In [ ]:
import os
import shutil
from pathlib import Path

repo_data = REPO / 'data'
drive_data = Path(DRIVE_DATA)
repo_data.mkdir(exist_ok=True)

for name in ['raw', 'processed']:
    src = drive_data / name
    dst = repo_data / name
    if not src.exists():
        raise FileNotFoundError(f'Missing Drive data folder: {src}')
    if dst.is_symlink() or dst.is_file():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)
    os.symlink(src, dst)
    print(dst, '->', src)

processed = repo_data / 'processed' / DATASET
required = [processed / 'train.parquet', processed / 'test.parquet', processed / 'metadata.parquet']
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
print('processed dataset ready:', processed)

## 5. Optional Smoke Run

Run this first if you want to verify paths, CUDA, and output schema before committing to the full job.

In [ ]:
if RUN_SMOKE_SENTIMENT:
    run(sys.executable, '-u', '-m', 'src.features.build_sentiment_features', '--dataset', DATASET, '--max-rows', '5000')
else:
    print('Skipping smoke sentiment run')


## 6. Full Sentiment Feature Build

Run this after the smoke run works. It overwrites the smoke artifacts at the end of the job.

In [ ]:
if RUN_FULL_SENTIMENT:
    run(sys.executable, '-u', '-m', 'src.features.build_sentiment_features', '--dataset', DATASET)
else:
    print('Skipping full sentiment run')


## 7. Validate Outputs

Check that the run used CUDA and produced the expected row count.

In [ ]:
import json
import pandas as pd
from pathlib import Path

af = Path('data/processed') / DATASET / 'advanced_features'
meta = json.loads((af / 'train_sentiment_meta.json').read_text())
sent = pd.read_parquet(af / 'train_sentiment.parquet')
users = pd.read_parquet(af / 'user_review_aggregates.parquet')
items = pd.read_parquet(af / 'item_review_aggregates.parquet')

print(json.dumps(meta, indent=2))
print('sentiment shape:', sent.shape)
print('user aggregates:', users.shape)
print('item aggregates:', items.shape)
print(sent['sentiment_label'].value_counts(dropna=False))
print(sent.head(10).to_string(index=False))

## 8. Optional Fixed-Alpha Evaluation Sweep

Set `RUN_ALPHA_SWEEP = True` in the controls cell to run this. By default it is capped for large datasets; remove caps only when you intentionally want final full metrics.

The sweep does not rebuild sentiment. It reuses processed data, sentiment aggregates, and embedding caches, then writes `metrics_alpha_sweep.json`.


In [ ]:
import json
from pathlib import Path

if RUN_ALPHA_SWEEP:
    processed = Path('data/processed') / DATASET
    all_rows = []
    for alpha in ALPHAS:
        print(f'\n=== evaluating alpha={alpha} ===', flush=True)
        cmd = [
            sys.executable, '-u', '-m', 'src.evaluation.evaluate',
            '--dataset', DATASET,
            '--no-knn',
            '--advanced',
            '--alpha', str(alpha),
        ]
        if MAX_EVAL_USERS is not None:
            cmd.extend(['--max-eval-users', str(MAX_EVAL_USERS)])
        if MAX_TEST_ROWS is not None:
            cmd.extend(['--max-test-rows', str(MAX_TEST_ROWS)])
        run(*cmd)

        rows = json.loads((processed / 'metrics.json').read_text())
        for row in rows:
            row['alpha'] = alpha
        all_rows.extend(rows)

    out = processed / 'metrics_alpha_sweep.json'
    out.write_text(json.dumps(all_rows, indent=2))
    print(f'Wrote {out}')
else:
    print('Skipping alpha sweep evaluation')


## 9. Bring Artifacts Back Local

Because `data/processed` points to Drive, the artifacts are already persisted there. Copy or sync this folder back to your local repo before running evaluation locally:

```text
data/processed/<dataset>/advanced_features/
```

Do not commit those artifacts.